# Defect / No-Defect Classification Experiments

This notebook outlines a series of data preparation and training experiments to build a binary classifier for grayscale images organized as:

```
dataset/
  train/
    defect/
    no_defect/
  val/
    defect/
    no_defect/
  test/
    defect/
    no_defect/
```

Run the notebook top-to-bottom to try different augmentations and hyperparameters. Adjust paths as needed for your environment.


In [ ]:
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.transforms import functional as TF

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_ROOT = Path("dataset")
BATCH_SIZE = 32
NUM_WORKERS = 2


## Dataset utilities

In [ ]:
def count_classes(split: str) -> Dict[str, int]:
    # Return class counts for a split using the folder names.
    ds = datasets.ImageFolder(DATA_ROOT / split)
    counts = {ds.classes[idx]: 0 for idx in range(len(ds.classes))}
    for _, label in ds.samples:
        counts[ds.classes[label]] += 1
    return counts

for split in ["train", "val", "test"]:
    split_path = DATA_ROOT / split
    if split_path.exists():
        print(split, count_classes(split))
    else:
        print(f"{split_path} not found; create this split to compute counts.")


## Data preparation experiments

Define a set of transformation pipelines to compare. The baselines keep images grayscale but replicate channels to use ImageNet-pretrained models. Additional pipelines apply histogram equalization, gamma adjustment, or heavier augmentation to simulate different capture conditions.


In [ ]:
# Helper transform to keep grayscale inputs compatible with pretrained RGB models
make_grayscale_rgb = transforms.Lambda(lambda img: TF.to_grayscale(img, num_output_channels=3))

transform_experiments: Dict[str, transforms.Compose] = {
    "baseline": transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        make_grayscale_rgb,
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
    ]),
    "histogram_eq": transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.RandomAutocontrast(p=0.9),
        make_grayscale_rgb,
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
    ]),
    "gamma_and_flip": transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.RandomAdjustSharpness(1.5)], p=0.5),
        transforms.RandomApply([transforms.RandomAffine(degrees=10, translate=(0.05, 0.05))], p=0.7),
        transforms.RandomApply([transforms.RandomAutocontrast()], p=0.7),
        make_grayscale_rgb,
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
    ]),
    "strong_aug": transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.1, contrast=0.2)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(5)], p=0.3),
        make_grayscale_rgb,
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
    ]),
}

# Validation/testing should remain deterministic
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    make_grayscale_rgb,
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
])


### Data loaders

In [ ]:
def get_dataloaders(train_transform_name: str) -> Tuple[DataLoader, DataLoader, DataLoader]:
    if train_transform_name not in transform_experiments:
        raise ValueError(f"Unknown transform set: {train_transform_name}")

    train_ds = datasets.ImageFolder(DATA_ROOT / "train", transform=transform_experiments[train_transform_name])
    val_ds = datasets.ImageFolder(DATA_ROOT / "val", transform=val_transform)
    test_ds = datasets.ImageFolder(DATA_ROOT / "test", transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


## Model definition

Use a pretrained ResNet-18 with a lightweight classification head. A dropout layer in the head can be toggled per-experiment to test regularization strength.


In [ ]:
def build_model(dropout: float = 0.0) -> nn.Module:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 2)
    )
    return model.to(device)


## Training and evaluation helpers

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = loss_fn(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def train_model(model: nn.Module, loaders: Tuple[DataLoader, DataLoader], epochs: int, lr: float, weight_decay: float) -> Dict[str, List[float]]:
    train_loader, val_loader = loaders
    loss_fn = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        scheduler.step()

        train_loss = running_loss / max(total, 1)
        train_acc = correct / max(total, 1)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f} acc={train_acc:.3f} | val_loss={val_loss:.4f} acc={val_acc:.3f}")
    return history


## Hyperparameter sweeps

Define experiment configurations to compare data preparation pipelines and optimizer settings. Keep the number of epochs small for quick iteration, then scale up the best configuration.


In [ ]:
        experiment_grid = [
            {"name": "baseline_lr1e-3", "transform": "baseline", "epochs": 5, "lr": 1e-3, "weight_decay": 1e-4, "dropout": 0.0},
            {"name": "hist_eq_dropout", "transform": "histogram_eq", "epochs": 6, "lr": 8e-4, "weight_decay": 5e-4, "dropout": 0.3},
            {"name": "gamma_flip_strong_reg", "transform": "gamma_and_flip", "epochs": 8, "lr": 5e-4, "weight_decay": 1e-3, "dropout": 0.4},
            {"name": "strong_aug_fast", "transform": "strong_aug", "epochs": 5, "lr": 1.5e-3, "weight_decay": 1e-4, "dropout": 0.2},
        ]

        results = []
        for cfg in experiment_grid:
            print("
=== Running", cfg["name"], "===")
            train_loader, val_loader, _ = get_dataloaders(cfg["transform"])
            model = build_model(dropout=cfg["dropout"])
            history = train_model(model, (train_loader, val_loader), epochs=cfg["epochs"], lr=cfg["lr"], weight_decay=cfg["weight_decay"])
            best_val_acc = max(history["val_acc"])
            results.append({"config": cfg, "best_val_acc": best_val_acc, "history": history})

        print("
Summary (sorted by best val acc):")
        for res in sorted(results, key=lambda r: r["best_val_acc"], reverse=True):
            cfg = res["config"]
            print(f"{cfg['name']}: val_acc={res['best_val_acc']:.3f}, transform={cfg['transform']}, lr={cfg['lr']}, wd={cfg['weight_decay']}, dropout={cfg['dropout']}")

        with open("experiment_results.json", "w") as f:
            json.dump(results, f, indent=2)
            print("Saved raw histories to experiment_results.json")


## Evaluate the best model on the test set

After inspecting validation performance, reload and re-train the best configuration with additional epochs if desired. Then, run on the held-out test split.


In [ ]:
# Example: pick the top-performing configuration from the previous run
if results:
    best_cfg = max(results, key=lambda r: r["best_val_acc"])["config"]
    print("Retraining best config for test evaluation:", best_cfg)
    train_loader, val_loader, test_loader = get_dataloaders(best_cfg["transform"])
    model = build_model(dropout=best_cfg["dropout"])
    _ = train_model(model, (train_loader, val_loader), epochs=max(10, best_cfg["epochs"]), lr=best_cfg["lr"], weight_decay=best_cfg["weight_decay"])
    test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss())
    print(f"Test set: loss={test_loss:.4f}, acc={test_acc:.3f}")
else:
    print("No results found. Run the sweep cell first.")


## Next steps

- Increase epochs and try lower learning rates once overfitting is under control.
- Add focal loss if class imbalance is significant.
- Export the trained model with `torch.save` for deployment or convert to TorchScript.
- Track runs with tools like TensorBoard or Weights & Biases for deeper diagnostics.
